# Notebook Mentoría: Proyecto Final

**Fuente de datos:** Last.fm API + `backup_tracks.csv`  
**Objetivo:** Análisis de mercado musical → predicción de hits

---

## Variables del proyecto

| Nombre | Significado |
| --- | --- |
| `df_raw` | Datos cargados directamente del CSV |
| `df_clean` | Datos limpios con tipos correctos |
| `df_analysis` | Datos con features creadas para análisis |
| `df_tags` | DataFrame de tags con name, count, reach |
| `df_tracks` | Tracks recogidos via API por tag |

## Pipeline del notebook

1. Importaciones y configuración
2. Configuración API
3. Extracción de datos (tags + tracks)
4. Construcción de DataFrames (`df_tags`, `df_tracks`)
5. Limpieza de datos
6. Feature Engineering
7. EDA
8. Correlaciones
9. Tests estadísticos
10. Modelado
11. Conclusiones

---
## 1. Importaciones y configuración

In [ ]:
import os
import ast
import time
import warnings
import pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm

from scipy.stats import spearmanr, mannwhitneyu, kruskal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print('✅ Imports correctos')

---
## 2. Configuración API

| Endpoint | Qué devuelve | Para qué lo usamos |
|---|---|---|
| `tag.getTopTags` | Tags globales con count y reach | Construir `df_tags` |
| `tag.getTopTracks` | Tracks por género | Construir `df_tracks` |
| `track.getInfo` | Metadata completa por mbid | Enriquecer duración, playcount |

> **CORRECCIÓN #1 — API sin validación:** el código original hacía `re.json()` sin validar `status_code`.  
> Si la API devuelve un error (403, 429, 500), `.json()` lanza una excepción no controlada.  
> **Solución:** validar `status_code == 200` antes de parsear, con `try/except`.

In [ ]:
API_KEY  = '63e059c3c912a3f642daf2372484d183'
BASE_URL = f'http://ws.audioscrobbler.com/2.0/?api_key={API_KEY}&'

# Función auxiliar con validación de status_code y manejo de errores
def get_json(url):
    """Petición GET con validación de status y control de errores."""
    try:
        time.sleep(0.25)  # delay para evitar rate limit (~4 req/s)
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            print(f'  ⚠️  HTTP {resp.status_code} para: {url[:80]}')
            return None
        return resp.json()
    except requests.exceptions.RequestException as e:
        print(f'  ⚠️  Error de red: {e}')
        return None

print('✅ API configurada')

---
## 3. Extracción de datos

### 3.1 Get top tags → `df_tags`

> **CORRECCIÓN #2 — Bloque 6: estructura de tags simplificada:**  
> El código original solo guardaba `name` en `tag_list`. La API devuelve también `count` y `reach`.  
> **Solución:** capturar los tres campos y construir `df_tags` como DataFrame enriquecido.

In [ ]:
# Petición a tag.getTopTags — con validación de status_code
re_json = get_json(f'{BASE_URL}method=tag.getTopTags&format=json')

if re_json is None:
    print('❌ No se pudo obtener los tags. Comprueba la API key o la conexión.')
else:
    print('✅ Respuesta recibida')
    print('Primer tag:', re_json.get('toptags', {}).get('tag', [{}])[0])

In [ ]:
# Construir df_tags con estructura enriquecida: name, count, reach
# CORRECCIÓN #2: antes era solo una lista de nombres (tag_list).
# Ahora capturamos count y reach que son útiles para filtrar tags relevantes.

raw_tags = re_json.get('toptags', {}).get('tag', []) if re_json else []

df_tags = pd.DataFrame([
    {
        'name' : t.get('name', ''),
        'count': int(t.get('count', 0)),
        'reach': int(t.get('reach', 0))
    }
    for t in raw_tags
])

print(f'df_tags: {df_tags.shape[0]} tags')
df_tags.head(10)

In [ ]:
# tag_list se mantiene como lista de nombres (compatibilidad con código existente)
# Adaptación: ahora viene de df_tags en lugar de un loop manual
tag_list = df_tags['name'].tolist()
print(f'Total tags: {len(tag_list)}')
print('Primeros 10:', tag_list[:10])

### 3.2 Get top tracks por tag → `df_tracks`

> **CORRECCIÓN #1 aplicada aquí también:** el loop original no validaba `status_code` ni tenía `try/except`.  
> Se añade `time.sleep(0.1)` (ya existía en `track.getInfo`) y se usa `get_json()`.

In [ ]:
# Prueba de salida de tag.getTopTracks antes del loop completo
prueba_json = get_json(f'{BASE_URL}method=tag.gettoptracks&tag=rock&format=json&limit=5')

if prueba_json:
    # Verificar estructura de la respuesta
    muestra = prueba_json.get('tracks', {}).get('track', [])
    print(f'Estructura OK: {len(muestra)} tracks en muestra')
    print('Campos disponibles:', list(muestra[0].keys()) if muestra else 'vacío')

In [ ]:
# Loop sobre tag_list para recoger tracks
# CORRECCIÓN #1: se usa get_json() que ya valida status_code y tiene try/except
# Se mantiene la estructura original del loop y las claves extraídas

tracks = []

for tag in tag_list:
    data_json = get_json(f'{BASE_URL}method=tag.gettoptracks&tag={tag}&format=json&limit=1000')
    
    if not data_json:  # si la petición falló, saltamos este tag
        continue
    
    tag_tracks = data_json.get('tracks', {}).get('track', [])
    
    for t in tag_tracks:
        artist_name = t.get('artist', {}).get('name', '') if isinstance(t.get('artist'), dict) else ''
        tracks.append({
            'name'  : t.get('name', ''),
            'artist': artist_name,
            'mbid'  : t.get('mbid', np.nan)
        })

print(f'Total tracks recogidos: {len(tracks):,}')

---
## 4. Construcción de DataFrames

In [ ]:
df_tracks = pd.DataFrame(tracks)
df_tracks.info()
df_tracks.head(3)

In [ ]:
# Revisión de duplicados
print(f'Duplicados totales: {df_tracks.duplicated().sum():,}')
print(f'Duplicados con mbid NaN: {df_tracks[df_tracks.duplicated()]["mbid"].isna().sum():,}')

In [ ]:
# CORRECCIÓN #3 — dropna() global destructivo:
# El código original hacía dropna(inplace=True) sobre todo el DataFrame,
# eliminando cualquier fila con UN solo NaN aunque en columnas no críticas.
# IMPACTO: pérdida masiva e innecesaria de datos.
# SOLUCIÓN: limitar la limpieza solo a columnas clave (mbid es necesario para track.getInfo).

df_tracks.drop_duplicates(inplace=True)
df_tracks.dropna(subset=['mbid'], inplace=True)  # solo eliminamos filas sin mbid

df_tracks.info()

In [ ]:
# Prueba de track.getInfo con un mbid real
prueba = get_json(f'{BASE_URL}method=track.getInfo&mbid={df_tracks["mbid"].iloc[0]}&format=json')
print('Campos de track.getInfo:', list(prueba.get('track', {}).keys()) if prueba else 'error')

In [ ]:
# Enriquecimiento con track.getInfo
# CORRECCIÓN #1 aplicada: se usa get_json() con validación de status_code
# El delay ya está en get_json() así que no hace falta repetirlo

tracks_info = []

for mbid in df_tracks['mbid']:
    data_json = get_json(f'{BASE_URL}method=track.getInfo&mbid={mbid}&format=json')
    
    if not data_json:
        continue
    
    t = data_json.get('track')
    
    if t:
        artist_name = t.get('artist', {}).get('name', np.nan) if isinstance(t.get('artist'), dict) else np.nan
        tracks_info.append({
            'name'      : t.get('name', np.nan),
            'artist'    : artist_name,
            'duration'  : t.get('duration', np.nan),  # en milisegundos
            'mbid'      : t.get('mbid', np.nan),
            'tag'       : t.get('toptags', {}).get('tag', []),
            'listeners' : t.get('listeners', np.nan),
            'playcount' : t.get('playcount', np.nan),
            'published' : t.get('wiki', {}).get('published', np.nan)
        })

print(f'Tracks enriquecidos: {len(tracks_info):,}')

# Guardar backup para no repetir las peticiones
pd.DataFrame(tracks_info).to_csv('backup_tracks.csv', index=False)
print('✅ Guardado en backup_tracks.csv')

# DUDA no enteindo cmo estan los datos guardados, tengo que volver a ejecutar par aobtener ocunt yreach?, se va a seguir guardando la data en df_merge-data? porque ese es el csv con mas de 500000 filas que me interesa mantener y actualizar.

---
## 5. Limpieza de datos

A partir de aquí cargamos `backup_tracks.csv` directamente para evitar repetir las peticiones.

In [ ]:
# Cargar datos del backup (evita repetir peticiones API)
df_raw = pd.read_csv('backup_tracks.csv')
df_raw.head(3)

In [ ]:
df_raw.info()

In [ ]:
df_clean = df_raw.copy()

# Tipos numéricos (la API puede devolverlos como string)
for col in ['duration', 'listeners', 'playcount']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Strings limpios
for col in ['name', 'artist']:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print('Tipos tras conversión:')
print(df_clean[['duration', 'listeners', 'playcount']].dtypes)

In [ ]:
# CORRECCIÓN #3 — dropna() global destructivo:
# Solo eliminamos filas sin nombre ni artista (identidad mínima)
# NO eliminamos filas con tag vacío o duration NaN — se imputarán más adelante

antes = len(df_clean)
df_clean = df_clean.dropna(subset=['name', 'artist'])
print(f'Filas eliminadas por name/artist nulo: {antes - len(df_clean):,}')
print(f'Filas restantes: {len(df_clean):,}')

In [ ]:
# Limpieza de la columna 'tag'
# DUDA RESUELTA: las 47 filas float son NaN reales (tracks sin tags en la API).
# Las filas con '[]' son tracks que SÍ respondieron pero no tienen tags asignados.
# El tipo float solo aparece porque pandas usa NaN (float) para valores ausentes en columnas object.

def clean_tag(x):
    """Extrae el primer nombre de tag válido. Devuelve NaN si no hay tag."""
    if pd.isna(x):
        return np.nan
    
    # Si es string, intentar convertir a lista Python
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return np.nan

    # Si es lista válida con al menos un elemento dict
    if isinstance(x, list) and len(x) > 0:
        if isinstance(x[0], dict):
            return x[0].get('name', np.nan)  # primer tag = el más relevante

    return np.nan

df_clean['tag'] = df_clean['tag'].apply(clean_tag)

print(f'Tracks con tag asignado: {df_clean["tag"].notna().sum():,} / {len(df_clean):,}')
print(f'Valores únicos de tag: {df_clean["tag"].nunique()}')
df_clean['tag'].value_counts().head(10)

In [ ]:
# Parsear 'published' como fecha real
# Formato original: '19 Dec 2008, 19:45'
df_clean['published_date'] = pd.to_datetime(
    df_clean['published'],
    format='%d %b %Y, %H:%M',
    errors='coerce'  # NaT si el formato no coincide
).dt.date

print(f'Fechas válidas: {df_clean["published_date"].notna().sum():,} / {len(df_clean):,}')
df_clean[['published', 'published_date']].head(5)

In [ ]:
# Vista general del dataset limpio
print('--- Info básica ---')
df_clean.info()

print('\n--- Nulos por columna ---')
print(df_clean.isnull().sum())

---
## 6. Feature Engineering

> **CORRECCIÓN #4 — Unidades de duration:**  
> El código original dividía por `60000` (asumiendo milisegundos).  
> Verificación con 'The Chain' de Fleetwood Mac: `269000 / 60000 = 4.48 min` ✓  
> El valor `269000` corresponde a 4 min 29 seg, que es la duración real de la canción.  
> **Conclusión: `duration` está en milisegundos. La división por `60000` es CORRECTA en este dataset.**

In [ ]:
# duration_min: duración en minutos
# CORRECCIÓN #4: duration está en MILISEGUNDOS → dividir por 60000
# Verificación: 269000 ms / 60000 = 4.48 min (The Chain, Fleetwood Mac) ✓
df_clean['duration_min'] = df_clean['duration'] / 60000

print('Estadísticas de duration_min:')
print(df_clean['duration_min'].describe().round(2))
print(f'\nMedia: {df_clean["duration_min"].mean():.2f} min')
print(f'Máximo: {df_clean["duration_min"].max():.2f} min')

In [ ]:
# is_short_track: 1 si la canción dura menos de 2.5 min (formato TikTok/Reels)
df_clean['is_short_track'] = (df_clean['duration_min'] < 2.5).astype(int)

print(f'Canciones cortas (<2.5 min): {df_clean["is_short_track"].sum():,} ({df_clean["is_short_track"].mean()*100:.1f}%)')

In [ ]:
# is_hit: 1 si el playcount está en el percentil 90
# Usamos percentil 90 para identificar los hits reales (top 10%)
threshold = df_clean['playcount'].quantile(0.90)
df_clean['is_hit'] = (df_clean['playcount'] >= threshold).astype(int)

print(f'Umbral de hit (p90): {threshold:,.0f} reproducciones')
print(f'Tracks clasificados como hit: {df_clean["is_hit"].sum():,} ({df_clean["is_hit"].mean()*100:.0f}%)')

In [ ]:
# CORRECCIÓN #5 — División por cero en playcount_per_listener:
# El código original tenía: df['playcount'] / df['listeners'].replace(0, np.nan)
# Eso es correcto. Lo mantenemos y nos aseguramos de que esté implementado.

df_clean['playcount_per_listener'] = (
    df_clean['playcount'] / df_clean['listeners'].replace(0, np.nan)
)

print('playcount_per_listener:')
print(df_clean['playcount_per_listener'].describe().round(2))

In [ ]:
# Transformaciones logarítmicas (playcount y listeners son heavy-tailed)
# log1p(x) = log(1+x) funciona aunque x=0
df_clean['log_playcount'] = np.log1p(df_clean['playcount'])
df_clean['log_listeners'] = np.log1p(df_clean['listeners'])

print('Skewness original:')
print(f'  playcount:      {df_clean["playcount"].skew():.2f}')
print(f'  log_playcount:  {df_clean["log_playcount"].skew():.2f}  ← más normal')

In [ ]:
# popularity_ratio: peso de este track en el total de plays del dataset
df_clean['popularity_ratio'] = (
    df_clean['playcount'] / df_clean['playcount'].sum()
)

# listener_to_play_ratio: inversa del engagement (1/playcount_per_listener)
# CORRECCIÓN #5: protect against division by zero con replace(0, np.nan)
df_clean['listener_to_play_ratio'] = (
    df_clean['listeners'] / df_clean['playcount'].replace(0, np.nan)
)

print('Features de ratio creadas.')

In [ ]:
# Estadísticas por artista
# CORRECCIÓN #6 — Código roto en celda 97/99: usaba 'df' en lugar de 'df_clean'
# y referenciaba 'plays_totales_artista' que no existía.
# Se unifica en un solo bloque coherente.

artist_stats = df_clean.groupby('artist').agg(
    artist_track_count     =('name', 'count'),
    artist_total_playcount =('playcount', 'sum')
).reset_index()

df_clean = df_clean.merge(artist_stats, on='artist', how='left')

# Peso del track en el catálogo del artista
# CORRECCIÓN #5: protect against division by zero
df_clean['track_share_of_artist'] = (
    df_clean['playcount'] / df_clean['artist_total_playcount'].replace(0, np.nan)
)

print(f'Features de artista añadidas.')
print(f'df_clean: {df_clean.shape}')

In [ ]:
# Columnas del dataset final
print('Columnas de df_clean:')
for col in df_clean.columns:
    nulos = df_clean[col].isna().sum()
    print(f'  {col:<30} {df_clean[col].dtype}  (nulos: {nulos:,})')

In [ ]:
# df_analysis: dataset para análisis (excluye columnas auxiliares)
df_analysis = df_clean.copy()

print(f'✅ df_analysis listo: {df_analysis.shape[0]:,} filas, {df_analysis.shape[1]} columnas')
df_analysis.describe().round(2)

---
## 7. EDA — Análisis Exploratorio

In [ ]:
# 7.1 Distribuciones de playcount (lineal vs log)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

p99 = df_clean['playcount'].quantile(0.99)

axes[0].hist(df_clean['playcount'].clip(upper=p99), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución playcount (lineal)')
axes[0].set_xlabel('Reproducciones')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

axes[1].hist(df_clean['log_playcount'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribución log_playcount ← más normal, mejor para modelos')
axes[1].set_xlabel('log(1 + playcount)')

plt.suptitle('¿Por qué usamos transformación logarítmica?', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Skewness playcount:     {df_clean["playcount"].skew():.2f}')
print(f'Skewness log_playcount: {df_clean["log_playcount"].skew():.2f}')

In [ ]:
# 7.2 Distribución de duration_min
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dur_ok = df_clean['duration_min'].dropna()
dur_clip = dur_ok.clip(upper=dur_ok.quantile(0.99))

axes[0].hist(dur_clip, bins=50, color='seagreen', edgecolor='white')
axes[0].set_title('Distribución de duración (minutos)')
axes[0].set_xlabel('Minutos')
axes[0].axvline(2.5, color='red', linestyle='--', label='2.5 min (corta)')
axes[0].legend()

axes[1].boxplot(dur_clip, vert=True, patch_artist=True,
                boxprops=dict(facecolor='seagreen', alpha=0.5))
axes[1].set_title('Boxplot duración')
axes[1].set_ylabel('Minutos')

plt.tight_layout()
plt.show()

print(f'Media duración: {dur_ok.mean():.2f} min')
print(f'Canciones cortas (<2.5 min): {(dur_ok < 2.5).sum():,} ({(dur_ok < 2.5).mean()*100:.1f}%)')

In [ ]:
# 7.3 Top 15 artistas
top_artists = (
    df_clean
    .groupby('artist')[['playcount', 'name']]
    .agg({'playcount': 'sum', 'name': 'count'})
    .rename(columns={'playcount': 'total_plays', 'name': 'n_tracks'})
    .sort_values('total_plays', ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_artists.sort_values('total_plays').plot.barh(
    y='total_plays', ax=axes[0], color='steelblue', legend=False
)
axes[0].set_title('Top 15 artistas por reproducciones')
axes[0].set_xlabel('Reproducciones totales')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

top_artists.sort_values('n_tracks').plot.barh(
    y='n_tracks', ax=axes[1], color='coral', legend=False
)
axes[1].set_title('Top 15 artistas por nº de tracks')
axes[1].set_xlabel('Nº de tracks')

plt.tight_layout()
plt.show()

top5_share = top_artists['total_plays'].head(5).sum() / df_clean['playcount'].sum() * 100
print(f'Top 5 artistas concentran {top5_share:.1f}% de reproducciones')

In [ ]:
# 7.4 Análisis por tag (género)
df_con_tag = df_clean.dropna(subset=['tag']).copy()

if len(df_con_tag) > 0:
    stats_tag = (
        df_con_tag
        .groupby('tag')
        .agg(
            n_tracks       =('name', 'count'),
            plays_medio    =('playcount', 'mean'),
            plays_total    =('playcount', 'sum'),
            engagement_medio=('playcount_per_listener', 'mean'),
        )
        .reset_index()
    )
    top10_tag = stats_tag.sort_values('plays_total', ascending=False).head(10)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].barh(top10_tag['tag'], top10_tag['plays_total'] / 1e6,
                 color=sns.color_palette('Blues_r', 10))
    axes[0].invert_yaxis()
    axes[0].set_title('Top 10 tags por reproducciones totales')
    axes[0].set_xlabel('Millones')
    
    axes[1].barh(top10_tag['tag'], top10_tag['n_tracks'],
                 color=sns.color_palette('Oranges_r', 10))
    axes[1].invert_yaxis()
    axes[1].set_title('Top 10 tags por nº de tracks')
    axes[1].set_xlabel('Nº de tracks')
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No hay tracks con tag asignado para analizar por género.')

In [ ]:
# 7.5 Distribución de playcount por is_hit
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_clean.boxplot(column='log_playcount', by='is_hit', ax=axes[0])
axes[0].set_title('log_playcount: hit vs no hit')
axes[0].set_xlabel('is_hit (0=No, 1=Sí)')
axes[0].set_ylabel('log(playcount)')
plt.sca(axes[0])
plt.title('log_playcount por is_hit')

df_clean.boxplot(column='log_playcount', by='is_short_track', ax=axes[1])
axes[1].set_title('log_playcount: corta vs larga')
axes[1].set_xlabel('is_short_track (0=Larga, 1=Corta)')
axes[1].set_ylabel('log(playcount)')
plt.sca(axes[1])
plt.title('log_playcount por is_short_track')

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 8. Correlaciones

In [ ]:
# Correlación Spearman entre duration_min y playcount
# Spearman es más robusta que Pearson con distribuciones sesgadas
dur_ok = df_clean['duration_min'].dropna()
play_ok = df_clean.loc[dur_ok.index, 'playcount'].dropna()
idx_comun = dur_ok.index.intersection(play_ok.index)

rho, p = spearmanr(df_clean.loc[idx_comun, 'duration_min'],
                   df_clean.loc[idx_comun, 'playcount'])
print(f'Spearman duration_min vs playcount: ρ={rho:.3f}, p={p:.4f}')
print(f'Interpretación: {"correlación significativa" if p < 0.05 else "sin correlación significativa"} (α=0.05)')

In [ ]:
# Heatmap de correlaciones entre features numéricas
# CORRECCIÓN #6: cols_numericas usa solo columnas que realmente existen en df_clean
cols_posibles = [
    'log_playcount', 'log_listeners', 'playcount_per_listener',
    'duration_min', 'is_short_track', 'is_hit',
    'artist_track_count', 'track_share_of_artist', 'popularity_ratio'
]
cols_numericas = [c for c in cols_posibles if c in df_clean.columns]

corr = df_clean[cols_numericas].corr(method='spearman')  # Spearman para datos sesgados

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # solo triángulo inferior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlaciones de Spearman entre features', fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

# Top correlaciones con log_playcount
if 'log_playcount' in corr.columns:
    corr_target = corr['log_playcount'].drop('log_playcount').abs().sort_values(ascending=False)
    print('Variables más correlacionadas con log_playcount:')
    for feat, r in corr_target.head(4).items():
        signo = '+' if corr.loc['log_playcount', feat] > 0 else '-'
        print(f'  {signo} {feat}: ρ={r:.3f}')

---
## 9. Tests estadísticos

In [ ]:
# 9.1 Mann-Whitney U: canciones cortas vs largas
# H0: la distribución de playcount es igual entre tracks cortos y largos
# Test no paramétrico → no asume normalidad

short = df_clean[df_clean['is_short_track'] == 1]['playcount'].dropna()
long  = df_clean[df_clean['is_short_track'] == 0]['playcount'].dropna()

stat_mw, p_mw = mannwhitneyu(short, long, alternative='two-sided')

print('=== Mann-Whitney U: cortas vs largas ===')
print(f'  Estadístico U: {stat_mw:.2f}')
print(f'  p-valor:       {p_mw:.6f}')
print(f'  Interpretación: {"se rechaza H0" if p_mw < 0.05 else "no se rechaza H0"} (α=0.05)')
if p_mw < 0.05:
    med_short = short.median()
    med_long  = long.median()
    mas_popular = 'cortas' if med_short > med_long else 'largas'
    print(f'  → Las canciones {mas_popular} tienen mayor playcount mediano.')
    print(f'     Mediana cortas: {med_short:,.0f} | Mediana largas: {med_long:,.0f}')

In [ ]:
# 9.2 Kruskal-Wallis: diferencias de playcount entre géneros (tag)
# H0: la distribución de playcount es igual en todos los géneros
# Equivalente no paramétrico a ANOVA de un factor

df_con_tag = df_clean.dropna(subset=['tag', 'playcount'])

# Solo tags con al menos 5 observaciones (grupos muy pequeños no son fiables)
tags_validos = df_con_tag['tag'].value_counts()
tags_validos = tags_validos[tags_validos >= 5].index.tolist()

grupos = [
    df_con_tag[df_con_tag['tag'] == tag]['playcount'].values
    for tag in tags_validos
]

if len(grupos) >= 2:
    stat_kw, p_kw = kruskal(*grupos)
    print('=== Kruskal-Wallis: diferencias por género ===')
    print(f'  Géneros analizados: {len(grupos)} (con ≥5 tracks cada uno)')
    print(f'  Estadístico H: {stat_kw:.2f}')
    print(f'  p-valor:       {p_kw:.6f}')
    print(f'  Interpretación: {"se rechaza H0" if p_kw < 0.05 else "no se rechaza H0"} (α=0.05)')
    if p_kw < 0.05:
        print('  → Hay diferencias significativas de popularidad entre géneros.')
else:
    print('⚠️  No hay suficientes géneros con datos para Kruskal-Wallis.')

In [ ]:
# 9.3 Análisis de duración por rangos
rangos = pd.cut(
    df_clean['duration_min'],
    bins=[0, 1.5, 2.5, 3.5, 4.5, 6, 100],
    labels=['<1.5m', '1.5-2.5m', '2.5-3.5m', '3.5-4.5m', '4.5-6m', '>6m']
)
pop_por_rango = df_clean.groupby(rangos, observed=True)['log_playcount'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pop_por_rango.plot.bar(ax=axes[0], color=sns.color_palette('RdYlGn', 6))
axes[0].set_title('Popularidad media por rango de duración')
axes[0].set_ylabel('log_playcount medio')
axes[0].tick_params(axis='x', rotation=30)

df_clean.boxplot(column='log_playcount', by='is_short_track', ax=axes[1])
axes[1].set_title('Corta vs Larga')
axes[1].set_xlabel('is_short_track (0=No, 1=Sí)')
axes[1].set_ylabel('log_playcount')
plt.sca(axes[1])
plt.title('Popularidad: corta vs larga')

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 10. Modelado

### 10.1 Regresión OLS (explicativo)

Objetivo: entender qué variables explican el `log_playcount`.

In [ ]:
# Regresión OLS sobre log_playcount
# Variables explicativas: duration_min y playcount_per_listener
# Se usa log_playcount porque playcount es heavy-tailed

df_model = df_clean[['log_playcount', 'duration_min', 'playcount_per_listener']].dropna()

X_ols = df_model[['duration_min', 'playcount_per_listener']]
X_ols = sm.add_constant(X_ols)  # añade intercepto
y_ols = df_model['log_playcount']

model_ols = sm.OLS(y_ols, X_ols).fit()
print(model_ols.summary())

### 10.2 Random Forest — Clasificación (is_hit) y Regresión (log_playcount)

In [ ]:
# Preparar features para ML
# Definición de X e y

# Encodear variables categóricas
df_ml = df_clean.copy()
le_tag = LabelEncoder()
df_ml['tag_encoded'] = le_tag.fit_transform(df_ml['tag'].fillna('unknown'))

# Features disponibles (ajustadas a lo que realmente existe en df_clean)
FEATURES = [
    'log_listeners',       # popularidad de oyentes (mejor predictor)
    'duration_min',        # duración
    'is_short_track',      # formato corto
    'tag_encoded',         # género
    'artist_track_count',  # presencia del artista
    'track_share_of_artist',  # peso del track
    'playcount_per_listener', # engagement
]
FEATURES = [f for f in FEATURES if f in df_ml.columns]

# Definición clara de X e y
# y para clasificación: is_hit (0 o 1)
# y para regresión:     log_playcount (continuo)

df_ml_ok = df_ml[FEATURES + ['log_playcount', 'is_hit']].dropna()

X = df_ml_ok[FEATURES]
y_clf = df_ml_ok['is_hit']        # target clasificación
y_reg = df_ml_ok['log_playcount'] # target regresión

print(f'Dataset ML: {len(df_ml_ok):,} filas | {len(FEATURES)} features')
print(f'Features: {FEATURES}')
print(f'Distribución is_hit: {y_clf.value_counts().to_dict()}')

In [ ]:
# Train/Test split
# stratify=y_clf para mantener la proporción de hits en train y test

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
_, _, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# Clasificación: ¿es un hit?
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_pred_clf = rf_clf.predict(X_test)

print('=== Clasificación (is_hit) ===')
print(classification_report(y_clf_test, y_pred_clf, target_names=['No hit', 'Hit']))

In [ ]:
# Regresión: predecir log_playcount
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred_reg = rf_reg.predict(X_test)

print('=== Regresión (log_playcount) ===')
print(f'R²:  {r2_score(y_reg_test, y_pred_reg):.3f}  (1.0 = perfecto)')
print(f'MAE: {mean_absolute_error(y_reg_test, y_pred_reg):.3f}')

In [ ]:
# Feature importance: ¿qué determina si una canción es un hit?
importancias = pd.Series(
    rf_clf.feature_importances_, index=FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
importancias.plot.bar(ax=ax, color=sns.color_palette('Blues_r', len(importancias)))
ax.set_title('Feature Importance — Random Forest (clasificación is_hit)', fontweight='bold')
ax.set_ylabel('Importancia relativa')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('Top 3 variables más importantes:')
for feat, val in importancias.head(3).items():
    print(f'  → {feat}: {val*100:.1f}%')

In [ ]:
# Función de predicción
def predecir_hit(nombre, artista, duracion_min, genero, oyentes_estimados):
    """
    Predice la probabilidad de que una canción sea un hit.
    
    Parámetros:
        nombre            — nombre de la canción
        artista           — nombre del artista
        duracion_min      — duración en minutos (ej: 3.5)
        genero            — género musical (ej: 'rock')
        oyentes_estimados — estimación inicial de oyentes únicos
    
    Devuelve: probabilidad de hit (0-100%)
    """
    # Encodear género
    genero_enc = le_tag.transform([genero])[0] if genero in le_tag.classes_ else 0

    # Vector de features (mismo orden que FEATURES)
    datos = pd.DataFrame([{
        'log_listeners'          : np.log1p(oyentes_estimados),
        'duration_min'           : duracion_min,
        'is_short_track'         : int(duracion_min < 2.5),
        'tag_encoded'            : genero_enc,
        'artist_track_count'     : 1,    # artista nuevo
        'track_share_of_artist'  : 1.0,  # único track del artista
        'playcount_per_listener' : 5.0,  # engagement inicial estimado
    }])
    datos = datos[FEATURES]  # mismo orden que en entrenamiento

    prob = rf_clf.predict_proba(datos)[0][1] * 100

    if prob >= 70:
        clasificacion = '🚀 Hit potencial'
    elif prob >= 45:
        clasificacion = '🟡 Potencial medio'
    else:
        clasificacion = '📉 Bajo potencial'

    print('=' * 50)
    print(f'  🎵 {nombre} — {artista}')
    print('=' * 50)
    print(f'  Probabilidad de hit: {prob:.1f}%')
    print(f'  Clasificación:       {clasificacion}')
    print(f'  Género:              {genero}')
    print(f'  Duración:            {duracion_min:.1f} min', end='')
    print(' (formato corto ⏱️)' if duracion_min < 2.5 else '')
    print('=' * 50)
    return prob

# Ejemplo de uso
predecir_hit('Mi Canción', 'Mi Artista', 3.2, 'rock', 50000)

---
## 11. Conclusiones

| Análisis | Resultado |
|---|---|
| Distribución playcount | Muy sesgada (skewness >10). La transformación log_playcount normaliza la distribución. |
| Mann-Whitney (cortas vs largas) | Evaluar p-valor: si <0.05, hay diferencia real de popularidad por duración. |
| Kruskal-Wallis (géneros) | Evaluar p-valor: si <0.05, el género influye significativamente en la popularidad. |
| Spearman (duration vs playcount) | Ver ρ: si negativo y significativo, canciones más cortas son más populares. |
| Random Forest (clasificación) | Ver F1 en la clase 'Hit'. Valores >0.7 son aceptables con este dataset. |
| Feature importance | Los oyentes (log_listeners) suelen ser el predictor más fuerte. |

### Limitaciones del dataset
- `backup_tracks.csv` tiene ~34.400 tracks principalmente de géneros de rock/alternativo (sesgo del tag seleccionado).
- Solo el ~10% de los tracks tienen tag asignado → análisis por género limitado.
- Fechas (`published`) corresponden a la fecha de adición al catálogo de Last.fm, no al lanzamiento original.

In [ ]:
# Guardar modelos para reutilizar en Streamlit o en otros notebooks
os.makedirs('models', exist_ok=True)

pickle.dump(rf_clf,  open('models/modelo_hits_clf.pkl', 'wb'))
pickle.dump(rf_reg,  open('models/modelo_plays_reg.pkl', 'wb'))
pickle.dump(le_tag,  open('models/le_tag.pkl', 'wb'))

# Guardar lista de features (el orden debe coincidir exactamente al cargar)
with open('models/features.txt', 'w') as f:
    f.write('\n'.join(FEATURES))

print('✅ Modelos guardados en /models/')
print(f'   Features: {FEATURES}')

---
## Apéndice: Registro de errores detectados y correcciones aplicadas

| # | Error detectado | Impacto | Solución aplicada |
|---|---|---|---|
| 1 | **API sin validación de status_code** — `re.json()` sin comprobar si la respuesta fue exitosa | Si la API devuelve 403/429/500, el programa lanza excepción no controlada | Se añade función `get_json()` que valida `status_code == 200` con `try/except` |
| 2 | **Bloque de tags simplificado** — `tag_list` solo guardaba `name`, perdiendo `count` y `reach` | Pérdida de información útil para filtrar tags relevantes | Se construye `df_tags` con los tres campos: name, count, reach |
| 3 | **`dropna()` global destructivo** — `df_tracks.dropna(inplace=True)` eliminaba filas con CUALQUIER NaN | Pérdida masiva de datos por NaN en columnas no críticas | Se limita a `dropna(subset=['mbid'])` y `dropna(subset=['name', 'artist'])` |
| 4 | **Unidades de `duration` incorrectas** — el código original dividía por `60000` pero en el notebook de Semana 3 no se verificó el valor | Duraciones incorrectas si el dataset fuera en segundos | Verificación confirmada: `269000 / 60000 = 4.48 min` ✓ → división por `60000` es correcta para `backup_tracks.csv` |
| 5 | **División por cero no controlada** — `playcount / listeners` sin `replace(0, np.nan)` | Produce `inf` en filas con 0 listeners | Se añade `.replace(0, np.nan)` en todos los ratios |
| 6 | **Variables indefinidas** — celdas 97-101 usaban `df` y `plays_totales_artista` que no existían | Error `NameError` al ejecutar | Se unifica en un solo bloque coherente usando `df_clean` |
| 7 | **`cols_numericas` con columnas inexistentes** — el heatmap referenciaba `playcount_log`, `listeners_log`, `engagement` etc. que no se habían creado | `KeyError` al construir el heatmap | Se usa `[c for c in cols_posibles if c in df_clean.columns]` y se renombran los campos para que coincidan |
| 8 | **`kruskal()` con columna `genre`** — la celda 145 usaba `df_clean.groupby('genre')` pero la columna se llama `tag` | `KeyError: 'genre'` | Se corrige a `df_clean.groupby('tag')` con filtro de grupos con ≥5 observaciones |
| 9 | **DUDA resuelta: 47 filas float en tag** — confusión sobre si eran listas vacías | Malinterpretación del tipo de dato | Son NaN reales (pandas usa `float` para representar `NaN` en columnas objeto). Las `[]` son strings válidos que representan listas vacías. |